# Notebook 01: Data Exploration
Explore the synthetic Indian timetabling instances generated by `generate_indian_data.py`.

In [ ]:
import json, pathlib, pandas as pd, matplotlib.pyplot as plt
import sys; sys.path.insert(0, '../src')

In [ ]:
# Load instances
instances = {}
for p in sorted(pathlib.Path('../data/indian_synthetic').glob('*.json')):
    with open(p) as f:
        instances[p.stem] = json.load(f)
print(f'Loaded {len(instances)} instances:', list(instances.keys()))

## Instance Statistics

In [ ]:
rows = []
for name, inst in instances.items():
    rows.append({
        'instance': name,
        'n_courses': len(inst['courses']),
        'n_faculty': len(inst['faculty']),
        'n_rooms': len(inst['rooms']),
        'n_slots': len(inst['timeslots']),
        'n_groups': len(inst['student_groups']),
        'n_lab_courses': sum(1 for c in inst['courses'] if c['is_lab']),
        'mediums': list(set(c['medium'] for c in inst['courses']))
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

## Course Medium Distribution

In [ ]:
fig, axes = plt.subplots(1, len(instances), figsize=(5*len(instances), 4))
for ax, (name, inst) in zip(axes if len(instances)>1 else [axes], instances.items()):
    mediums = [c['medium'] for c in inst['courses']]
    unique, counts = zip(*pd.Series(mediums).value_counts().items())
    ax.bar(unique, counts, color=['#4C9BE8','#E87D4C','#4CE87D'])
    ax.set_title(f'{name}\nMedium Distribution')
    ax.set_ylabel('# Courses')
plt.tight_layout()
plt.savefig('01_medium_distribution.png', dpi=150)
plt.show()
print('Figure saved.')

## Room Type Distribution

In [ ]:
for name, inst in instances.items():
    rtypes = [r['type'] for r in inst['rooms']]
    print(f'{name}: {pd.Series(rtypes).value_counts().to_dict()}')

## Hard Constraint H7 (Medium Compliance) Analysis

In [ ]:
for name, inst in instances.items():
    faculty_map = {f['id']: f['medium'] for f in inst['faculty']}
    violations = sum(
        1 for c in inst['courses']
        if faculty_map.get(c['faculty_id'], '') != c['medium']
    )
    print(f'{name}: {violations}/{len(inst["courses"])} H7 violations in raw assignment')

## H8 Lab Batching Analysis

In [ ]:
for name, inst in instances.items():
    lab_courses = [c for c in inst['courses'] if c['is_lab']]
    print(f'{name}: {len(lab_courses)} lab courses (require {lab_courses[0]["lab_duration_slots"] if lab_courses else 0} consecutive slots)')